Install required packages

In [ ]:
!pip install -q sentence-transformers scikit-learn pandas numpy

Import the libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer

In [ ]:
from google.colab import files
uploaded = files.upload()

Load the dataset

In [ ]:
reviews = pd.read_csv("/content/thecourseforum_all_reviews (1).csv")

print("Dataset shape:", reviews.shape)
print("\nColumns:")
print(reviews.columns.tolist())

Check the two columns

In [ ]:
print("Missing review_text:", reviews["review_text"].isna().sum())
print("Missing difficulty_rating:", reviews["difficulty_rating"].isna().sum())

print("\nDifficulty rating distribution:")
print(reviews["difficulty_rating"].value_counts().sort_index())

print("\nFirst 3 reviews:")
print(reviews["review_text"].head(3))

Inspect the raw review structure

In [ ]:
for i in range(5):
    print(f"\n--- REVIEW {i} ---")
    print(reviews.loc[i, "review_text"])

Check the beginning and ending patterns across the dataset

In [ ]:
print("FIRST 10 REVIEW TEXTS:")
print("=" * 80)

for i in range(10):
    text = reviews.loc[i, "review_text"]
    print(f"\nReview {i}:")
    print(text[:150])
    print("...")
    print(text[-100:])

Clean review text

In [ ]:
import re

def clean_review(text):
    # remove the webpage heading containing the term, year, and average rating.
    # January is included so January-term reviews are cleaned too.
    text = re.sub(
        r'^(Fall|Spring|Summer|Winter|January)\s+\d{4}\s+\d+(?:\.\d+)?\s+AVERAGE\s*',
        '',
        text,
        flags=re.IGNORECASE
    )

    # remove the trailing vote count and posting date collected from the webpage.
    # keep the student's actual writing between the heading and footer.
    text = re.sub(
        r'\s+-?\d+\s+'
        r'(?:Jan\.?|Feb\.?|Mar\.?|Apr\.?|May|Jun\.?|Jul\.?|Aug\.?|'
        r'Sept?\.?|Oct\.?|Nov\.?|Dec\.?|January|February|March|April|'
        r'June|July|August|September|October|November|December)'
        r'\s+\d{1,2},\s+\d{4}\s*$',
        '',
        text,
        flags=re.IGNORECASE
    )

    # remove extra spaces from the beginning and end of the cleaned review.
    return text.strip()

# create a cleaned column while preserving the original review_text column.
reviews["cleaned_review"] = reviews["review_text"].apply(clean_review)

Check that the cleaning worked

In [ ]:
for i in range(5):
    print(f"\n--- CLEANED REVIEW {i} ---")
    print(reviews.loc[i, "cleaned_review"])

Create the features and target

In [ ]:
# Text feature
X = reviews["cleaned_review"]

# Target variable
y = reviews["difficulty_rating"]

print("Number of reviews:", len(X))
print("Number of labels:", len(y))

print("\nTarget distribution:")
print(y.value_counts().sort_index())

Create the 70/15/15 split

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

print("Training set size:", len(X_train))
print("Temporary set size:", len(X_temp))

print("\nTraining rating distribution:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTemporary rating distribution:")
print(y_temp.value_counts(normalize=True).sort_index())

Create validation and test sets

In [ ]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Training set size:", len(X_train))
print("Validation set size:", len(X_val))
print("Test set size:", len(X_test))

print("\nValidation rating distribution:")
print(y_val.value_counts(normalize=True).sort_index())

print("\nTest rating distribution:")
print(y_test.value_counts(normalize=True).sort_index())

Load MiniLM

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully.")

Generate training embeddings

In [ ]:
X_train_embeddings = model.encode(
    X_train.tolist(),
    show_progress_bar=True
)

print("Training embeddings shape:", X_train_embeddings.shape)

Generate validation embeddings

In [ ]:
X_val_embeddings = model.encode(
    X_val.tolist(),
    show_progress_bar=True
)

print("Validation embeddings shape:", X_val_embeddings.shape)

Generate test embeddings

In [ ]:
X_test_embeddings = model.encode(
    X_test.tolist(),
    show_progress_bar=True
)

print("Test embeddings shape:", X_test_embeddings.shape)

Final feature-engineering verification

In [ ]:
print("FINAL DATA CHECK")
print("-" * 40)

print("Training:")
print("  Embeddings:", X_train_embeddings.shape)
print("  Labels:", y_train.shape)

print("\nValidation:")
print("  Embeddings:", X_val_embeddings.shape)
print("  Labels:", y_val.shape)

print("\nTest:")
print("  Embeddings:", X_test_embeddings.shape)
print("  Labels:", y_test.shape)

Export and download embeddings

In [ ]:
import numpy as np
from google.colab import files

np.savez_compressed(
    "embeddings_split.npz",
    X_train=X_train_embeddings,
    X_val=X_val_embeddings,
    X_test=X_test_embeddings,
    y_train=y_train.to_numpy(),
    y_val=y_val.to_numpy(),
    y_test=y_test.to_numpy()
)

files.download("embeddings_split.npz")